In [ ]:
#Imports
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Any, Union, Literal, Callable
import shap
import platform
if platform.system() == "Linux":
    from sklearnex import patch_sklearn
    patch_sklearn()

from sklearn.pipeline import make_pipeline

from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold, cross_val_score, cross_validate
from sklearn.model_selection import RandomizedSearchCV

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import ElasticNetCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from xgboost import XGBRegressor

from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.base import clone

from sklearn.base import BaseEstimator
from sklearn.pipeline import Pipeline
from sklearn.linear_model._base import LinearModel
from sklearn.ensemble import BaseEnsemble
from xgboost import XGBModel 

In [ ]:
PmDataset = pd.DataFrame
ColumnNames = list[str]
ModelMetrics = tuple[float, float]

ModelCoefs = pd.DataFrame
ModelCoefsAbsMean = pd.DataFrame

TreeModel = Union[BaseEnsemble, XGBModel]
SklearnPipeline = Pipeline

PmSplitFeatures = pd.DataFrame
PmSplitTarget = pd.Series
PmSplitData = tuple[PmSplitFeatures, PmSplitFeatures, PmSplitTarget, PmSplitTarget]

In [ ]:
# ==========================================
# 1. DATA GENERATION
# ==========================================
def get_op_fn(op_fn_name:Literal["op_dtt", "op_aa", "op_dtt_custom", "op_custom"]="op_dtt"):
    def op_dtt_custom_fn(df:PmDataset):
        terms = {
            "OC": 0.35 * df["OC_ng_m3"],
            "EC": 0.20 * df["EC_ng_m3"],
            "Fe": 0.15 * df["Fe_ng_m3"],
            "Cu": 0.10 * df["Cu_ng_m3"],
            "Fe_x_Cu": 0.45 * df["Fe_ng_m3"] * df["Cu_ng_m3"],
            "Ec_x_Mn": 0.25 * df["EC_ng_m3"] * df["Mn_ng_m3"]
        }
        op = terms["OC"] + terms["EC"] + terms["Fe"] + terms["Cu"] + terms["Fe_x_Cu"] + terms["Ec_x_Mn"]
        return op, terms
    def op_dtt_fn(df:PmDataset):
        terms = {
            "OC": 0.35 * df["OC_ng_m3"],
            "EC": 0.20 * df["EC_ng_m3"],
            "Fe": 0.15 * df["Fe_ng_m3"],
            "Cu": 0.10 * df["Cu_ng_m3"],
            "Fe_x_Cu": 0.45 * df["Fe_ng_m3"] * df["Cu_ng_m3"]
        }
        op = terms["OC"] + terms["EC"] + terms["Fe"] + terms["Cu"] + terms["Fe_x_Cu"]
        return op, terms
    def op_aa_fn(df:PmDataset):
        terms = {
            "Cu": 0.05 * df["Cu_ng_m3"],
            "Mn": 0.65 * df["Mn_ng_m3"]
        }
        op = terms["Cu"] + terms["Mn"]
        return op, terms
    def op_custom_fn(df:PmDataset):
        terms = {
            "OC": 0.0085 * df["OC_ng_m3"],
            "EC": 0.011 * df["EC_ng_m3"],
            "OC_x_Ec": 0.00 * df["OC_ng_m3"] * df["EC_ng_m3"],
            "Fe": 0.10 * df["Fe_ng_m3"],
            "Cu": 1.0 * df["Cu_ng_m3"],
            "Fe_x_Cu": 0.015 * df["Fe_ng_m3"] * df["Cu_ng_m3"]
        }
        op = terms["OC"] + terms["EC"] + terms["Fe"] + terms["Cu"] + terms["Fe_x_Cu"] + terms["OC_x_Ec"]
        return op, terms
    op_fn_dict = {"op_dtt": op_dtt_fn, "op_aa": op_aa_fn, "op_dtt_custom": op_dtt_custom_fn, "op_custom": op_custom_fn}
    op_fn = op_fn_dict[op_fn_name]
    return op_fn

def make_data(
        n_samples=150, 
        n_sites=1, 
        op_dict:dict[str, Callable]={"OP_DTT_nmol_min_m3": get_op_fn("op_dtt"), "OP_AA_nmol_min_m3":get_op_fn("op_aa")}, 
        add_op_noise=True, 
        n_decoys=5, 
        rng_seed=1) -> tuple[PmDataset, tuple[ColumnNames]]:
    """
    Generates synthetic air quality concentration and oxidative potential (OP) data.
    """
    rng = np.random.default_rng(rng_seed)
    
    # Concentrations
    df = pd.DataFrame({
        "sample_id": [f"S{str(i+1).zfill(3)}" for i in range(n_samples)],
        "site": rng.choice([f"Urban_Site_{n}" for n in range(n_sites)], n_samples),
        "season": "Winter",
        "Fe_ng_m3": rng.lognormal(4.9, 0.4, n_samples),
        "Cu_ng_m3": rng.lognormal(2.3, 0.5, n_samples),
        "Mn_ng_m3": rng.lognormal(1.8, 0.45, n_samples),
        "EC_ng_m3": rng.lognormal(7.4, 0.15, n_samples),
        "OC_ng_m3": rng.lognormal(8.3, 0.04, n_samples),
    })
    str_cols_names = ["sample_id", "site", "season"]
    conc_cols_names = [c for c in df.columns if c not in str_cols_names]

    # OP
    op_cols_names = []
    for op_name, op_fn in op_dict.items():
        op_cols_names.append(op_name)
        op, _ = op_fn(df)
        if add_op_noise:
            df[op_name] = op + rng.normal(0, 0.4, n_samples)
        else:
            df[op_name] = op

    # Decoys
    decoy_cols_names = [f"Decoy{i+1}_ng_m3" for i in range(n_decoys)]
    decoy_data = np.abs(rng.normal(loc=10, scale=3, size=(len(df), n_decoys)))
    decoy_df = pd.DataFrame(decoy_data, columns=decoy_cols_names, index=df.index)
    
    first_target_idx = df.columns.get_loc(op_cols_names[0])
    df = pd.concat([
        df.iloc[:, :first_target_idx],
        decoy_df,
        df.iloc[:, first_target_idx:]
    ], axis=1)
        
    return df, (str_cols_names, conc_cols_names, op_cols_names, decoy_cols_names)

def poly_expand_add_data(df: PmDataset, col_names_to_expand:ColumnNames, op_cols_names:ColumnNames) -> tuple[PmDataset, ColumnNames]:
    """
    Generates degree-2 polynomial interaction features for the specified columns and appends them to the dataframe.

    Parameters
    ----------
    df
        The dataset to expand.
    col_names_to_expand
        List of column names to compute pairwise interactions for.
    op_cols_names
        List of target column names used for column ordering.

    Returns
    -------
    df
        The updated dataframe containing polynomial interactions.
    x_poly_clean_names
        List of the newly added polynomial feature names.
    """
    poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)

    x_poly_values_all = poly.fit_transform(df[col_names_to_expand])
    x_poly_names_all = poly.get_feature_names_out(col_names_to_expand)
    
    num_base_cols = len(col_names_to_expand)
    x_only_poly_values = x_poly_values_all[:, num_base_cols:]
    x_only_poly_names = x_poly_names_all[num_base_cols:]
    
    x_poly_clean_names = clean_poly_names(x_only_poly_names)
    x_poly_df = pd.DataFrame(x_only_poly_values, columns=x_poly_clean_names, index=df.index)

    first_target_idx = df.columns.get_loc(op_cols_names[0])

    df = pd.concat([
        df.iloc[:, :first_target_idx],
        x_poly_df,
        df.iloc[:, first_target_idx:]
    ], axis=1)

    return df, x_poly_clean_names

# ==========================================
# 2. Utils
# ==========================================

# General
def build_mean_coef_df(feature_names:ColumnNames, feature_values, col_name="coefficient") -> ModelCoefsAbsMean:
    """
    Feature values are average values (say coefficenits or average shap values in any format) -> in df format
    Constructs a DataFrame matching feature names to their coefficients or importances, sorted by magnitude.
    """
    df = pd.DataFrame({"feature": feature_names, col_name: feature_values})
    return df.sort_values(by=col_name, key=abs, ascending=False).reset_index(drop=True)

def clean_poly_names(names: ColumnNames) -> ColumnNames:
    """
    Replaces scikit-learn's default space-separated polynomial interaction names with an '_x_' separator.
    """
    return ["_x_".join([p.split("_")[0] for p in name.split(" ")]) for name in names]

def evaluate_model_performance(model:BaseEstimator, x_test:PmSplitFeatures, y_test:PmSplitTarget) -> ModelMetrics:
    """
    Calculates and returns the R^2 and RMSE scores for a given model.
    """
    y_pred = model.predict(x_test)
    r2 = r2_score(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    return r2, rmse

def cross_evaluate_model_performance(model:BaseEstimator, x_test:PmSplitFeatures, y_test:PmSplitTarget, n_splits=5, shuffle=True, random_state=1, scoring=["r2", "neg_root_mean_squared_error"]) -> tuple[ModelMetrics, pd.DataFrame]:
    kfold = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
    scores = cross_validate(model, x_test, y_test, scoring=scoring, cv=kfold)

    return scores["test_r2"].mean(), -1*scores["test_neg_root_mean_squared_error"].mean(), scores

def get_feature_rank(coef_df:ModelCoefsAbsMean, feature_name:str) -> int|None:
    """
    Retrieves the importance rank (1-based index) of a specific feature from a sorted coefficient dataframe.
    """
    if feature_name in coef_df["feature"].values:
        return coef_df[coef_df["feature"] == feature_name].index[0] + 1
    return None

def debug_op(df:PmDataset, op_fn:Callable) -> pd.DataFrame:
    op, terms = op_fn(df)
    rows = []
    for feature_name, feature_val in terms.items():
        rows.append({
            "feature": feature_name,
            "mean": feature_val.mean(),
            "std": feature_val.std()
        })

    return pd.DataFrame(rows)

# Lasso
def is_feature_nonzero(coef_df:ModelCoefsAbsMean, feature_name:str, col_name="coefficient") -> bool:
    """
    Checks whether a specific feature was assigned a non-zero coefficient.
    """
    if feature_name in coef_df["feature"].values:
        val = coef_df.loc[coef_df["feature"] == feature_name, col_name].values[0]
        return val != 0
    return False

# ==========================================
# 3. Tree explainers
# ==========================================

def compute_tree_importance(model:TreeModel, x_train:PmSplitFeatures, x_test:PmSplitFeatures, y_test:PmSplitTarget, permutation=True, perm_n_repeat=10, random_state=1):
    if permutation:
        perm = permutation_importance(model, x_test, y_test, n_repeats=perm_n_repeat, random_state=random_state)
        feature_importance = perm.importances_mean
    else:
        feature_importance = model.feature_importances_
        
    # FIX: Use x_test.columns if permutation is run on x_test, or pass a single x_data argument to this function.
    feature_importance_mean = build_mean_coef_df(x_test.columns, feature_importance, col_name="importance")

    return feature_importance, feature_importance_mean

def compute_global_shap(model:TreeModel, x_data:PmSplitFeatures) -> tuple[shap.Explanation, ModelCoefs, ModelCoefsAbsMean]:
    explainer = shap.TreeExplainer(model)
    shap_explanation = explainer(x_data)

    feature_names = x_data.columns
    shap_vals_df = pd.DataFrame(shap_explanation.values, columns=x_data.columns, index=x_data.index)
    shap_mean_abs_df = build_mean_coef_df(feature_names, shap_explanation.abs.mean(0).values, col_name="mean_abs_shap")
    
    return shap_explanation, shap_vals_df, shap_mean_abs_df

def compute_interaction_shap(model:TreeModel, x_data:PmSplitFeatures) ->tuple[shap.TreeExplainer, np.ndarray, pd.DataFrame]:
    '''
    Note, only shap fcuntino whose first return is explainer as there is no explanation object for interactino shap
    np.ndarray = ModelCoefs in this case is 3d matrix of np.ndarry (n_feature, n_features, n_samples)
    pd.Dataframe = ModelCoefsAbsMean is a 2nd matrix of df(n_features, n_features) where the values are average int shap values from all samples
    '''
    explainer = shap.TreeExplainer(model)
    shap_int_vals = explainer.shap_interaction_values(x_data)

    mean_abs_int_shap_vals = abs(shap_int_vals).mean(axis=0)
    feature_names = x_data.columns
    shap_int_mean_abs_df = pd.DataFrame(mean_abs_int_shap_vals, index=feature_names, columns=feature_names)

    return explainer, shap_int_vals, shap_int_mean_abs_df

def compute_partition_shap(model:TreeModel, x_data:PmSplitFeatures) -> tuple[shap.Explanation, ModelCoefs, ModelCoefsAbsMean]:
    """
    Computes SHAP values using PartitionExplainer, which relies on hierarchical 
    clustering of the input features (masker) rather than internal tree structures.
    """
    # 1. Create a partition masker based on the data distribution
    masker = shap.maskers.Partition(x_data, max_samples=250)
    
    # 2. Initialize the explainer using the model's predict method and the masker
    explainer = shap.PartitionExplainer(model.predict, masker)
    
    # 3. Compute the explanation
    shap_explanation = explainer(x_data)

    # 4. Format into DataFrames to match the rest of the pipeline
    feature_names = x_data.columns
    shap_vals_df = pd.DataFrame(shap_explanation.values, columns=feature_names, index=x_data.index)
    shap_mean_abs_df = build_mean_coef_df(feature_names, shap_explanation.abs.mean(0).values, col_name="mean_abs_shap")
    
    return shap_explanation, shap_vals_df, shap_mean_abs_df

def get_shap_int_diag(int_shap_vals:np.ndarray, x_train) -> pd.DataFrame:
    shap_int_diag_vals = np.diagonal(int_shap_vals, axis1=1, axis2=2)  # (n_samples, n_features)
    shap_int_diag_vals_df = pd.DataFrame(shap_int_diag_vals, columns=x_train.columns, index=x_train.index) # (n_samples, n_features)
    
    return shap_int_diag_vals_df

def transform_to_flit_shap(shap_vals_df:pd.DataFrame, y_train:PmSplitTarget) -> tuple[pd.DataFrame, Any]:
    importance = shap_vals_df.abs().mean()
    total_importance = importance.sum()
    global_intercept = y_train.mean()
    local_intercepts = (importance / total_importance) * global_intercept

    flit_vals_df = shap_vals_df + local_intercepts

    return flit_vals_df, local_intercepts

def flitify(glb_shap_vals_df:pd.DataFrame, int_shap_vals:np.ndarray, x_train:PmSplitFeatures, y_train:PmSplitTarget) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    sv, local_intercepts = transform_to_flit_shap(glb_shap_vals_df, y_train)
    me_raw = get_shap_int_diag(int_shap_vals, x_train)
    me = me_raw + local_intercepts
    ie = sv-me

    return sv, me, ie

def top_pairwise_shap_interaction(interaction_df, top_n=10):
    """
    Extracts the top N pairwise feature interactions from a SHAP interaction dataframe.
    """
    mask = np.triu(np.ones(interaction_df.shape), k=1).astype(bool)
    pairwise_series = interaction_df.where(mask).stack()
    top_pairs = pairwise_series.sort_values(ascending=False).head(top_n)
    top_pairs_df = top_pairs.reset_index()
    top_pairs_df.columns = ["Feature 1", "Feature 2", "Mean Abs SHAP Interaction"]
    return top_pairs_df

def plot_shap_interaction_heatmap(interaction_df, top_n=10):
    """
    Plots a heatmap for SHAP interactions of top_n.
    """
    feature_importance = interaction_df.sum(axis=1).sort_values(ascending=False)
    top_features = feature_importance.head(top_n).index
    top_interactions = interaction_df.loc[top_features, top_features] 
    
    fig, ax = plt.subplots(figsize=(10, 8)) # Slightly larger for better text fit

    sns.heatmap(top_interactions, annot=True, fmt=".3f", cmap="viridis", square=True, ax=ax)
    ax.set_title(f"Top {top_n} Features: SHAP Interaction Matrix\n(Diagonal = Main Effect, Off-Diagonal = Pairwise Interaction)")

    plt.tight_layout()
    plt.show()

# ==========================================
# 4. Helpers
# ==========================================

def random_split_helper(all_data:tuple[PmDataset, tuple[ColumnNames]], 
                        x_col_type: Literal["poly", "conc_decoy", "conc"],
                        y_col_name:ColumnNames, 
                        test_size=0.25, 
                        random_seed=1) -> PmSplitData:
    """
    Splits the dataset into training and testing sets based on the desired feature subsets.

    Parameters
    ----------
    all_data
        The tuple output from make_data_helper containing the dataframe and column lists.
    x_col_type
        Determines which columns form X: 'poly', 'conc_decoy', or 'conc'.
    y_col_name
        The target column to predict.
    test_size
        Proportion of the dataset to include in the test split.
    random_seed
        Random state for the train/test split.

    Returns
    -------
    x_train
    x_test
    y_train
    y_test
    """
    df, (str_cols_names, conc_cols_names, op_cols_names, decoy_cols_names, poly_cols_names_clean) = all_data

    x_train, x_test, y_train, y_test = train_test_split(df.drop(columns=op_cols_names), df[op_cols_names], test_size=test_size, random_state=random_seed)
    y_train, y_test = y_train[y_col_name], y_test[y_col_name]
    if x_col_type == "poly":
        x_train, x_test = x_train.drop(columns=str_cols_names), x_test.drop(columns=str_cols_names)
    elif x_col_type == "conc_decoy":
        x_train, x_test  = x_train[conc_cols_names+decoy_cols_names], x_test[conc_cols_names+decoy_cols_names]
    elif x_col_type == "conc":
        x_train, x_test = x_train[conc_cols_names], x_test[conc_cols_names]

    return x_train, x_test, y_train, y_test

def linear_model_helper(x_train:PmSplitFeatures, y_train:PmSplitTarget, model:LinearModel) -> tuple[SklearnPipeline, ModelCoefsAbsMean]:
    """
    Fits a linear model using a StandardScaler pipeline, evaluates it, and extracts the unscaled coefficients.
    """
    model = clone(model)

    pipeline = make_pipeline(StandardScaler(), model)
    pipeline.fit(x_train, y_train)

    scale_step = pipeline.named_steps["standardscaler"]
    lin_step = pipeline.steps[-1][1]
    coef_df = lin_step.coef_
    coef_mean_df_scaled = build_mean_coef_df(x_train.columns, coef_df/scale_step.scale_)

    return pipeline, coef_mean_df_scaled 

def tree_model_helper(x_train: PmSplitFeatures, y_train: PmSplitTarget, model: TreeModel|RandomizedSearchCV) -> tuple[TreeModel, object]:          
    model = clone(model)
    
    search = None
    if isinstance(model, (RandomizedSearchCV)):
        search = model
        search.fit(x_train, y_train)
        model = search.best_estimator_
    else:
        model.fit(x_train, y_train)

    return model, search

def benchmark(data:PmSplitData, 
              lin_models:dict[str,LinearModel], 
              tree_models:dict[str, TreeModel|RandomizedSearchCV],
              benchmark_type:Literal["Metrics", "Coefs", "Both"]="Both",
              metrics_fn:Callable=evaluate_model_performance,
              ):

    if benchmark_type == "Both":
        calc_coefs = True
        calc_metrics = True
    elif benchmark_type == "Coefs":
        calc_coefs = True
    elif benchmark_type == "metrics":
        calc_metrics = True
    
    x_train, x_test, y_train, y_test = data

    lin_results = {}
    tree_results = {}

    # Linear models
    for name, fn in lin_models.items():
        pipeline, coef_scaled = linear_model_helper(x_train, y_train, fn)
        lin_results[name] = {"model" : pipeline}
        if calc_metrics:
            metrics = metrics_fn(pipeline, x_test, y_test)
            lin_results[name] = {"metrics": metrics}
        if calc_coefs:
            lin_results[name]["coefs_scaled"] = coef_scaled    


    # Tree models
    for name, fn in tree_models.items():
        model, _ = tree_model_helper(x_train, y_train, fn)
        tree_results[name] = {"model" : model}
        if calc_metrics:
            metrics = metrics_fn(model, x_test, y_test)
            tree_results[name]["metrics"] = metrics
        if calc_coefs:
            _, glb_shap_explanation, glb_shap_mean_abs_df = compute_global_shap(model, x_train)
            tree_results[name]["glb_shap_mean_abs"] = glb_shap_mean_abs_df

            _, int_shap_vals, int_shap_mean_abs_df = compute_interaction_shap(model, x_train)
            interaction_shap_df = top_pairwise_shap_interaction(int_shap_mean_abs_df)
            tree_results[name]["int_shap_mean_abs_pairwise"] = interaction_shap_df

            _, _, ie = flitify(glb_shap_explanation, int_shap_vals, x_train, y_train)
            tree_results[name]["ie"] = ie.mean()

    # Metrics
    metrics_list = []
    all_results = {**lin_results, **tree_results}
    for name, result in all_results.items():
        metrics_list.append({"model": name, "r2": result["metrics"][0], "rmse": result["metrics"][1]})
    metrics_df = pd.DataFrame(metrics_list)

    importance_df = None
    # Linear coefficients
    for name, result in lin_results.items():
        df = result["coefs_scaled"]
        df = df.rename(columns={"coefficient": name})
        df = df[["feature", name]]

        if importance_df is None:
            importance_df = df
        else:
            importance_df = importance_df.merge(df, on="feature", how="outer")

    # Tree SHAP coefs
    for name, result in tree_results.items():
        # Get standard mean absolute SHAP
        df = result["glb_shap_mean_abs"].copy()
        df = df.rename(columns={"mean_abs_shap": f"{name}-SHAP"})
        df = df[["feature", f"{name}-SHAP"]]
        
        ie_df = result["ie"].reset_index()
        ie_df.columns = ["feature", f"{name}_FLIT-SHAP-IE"]
        df = df.merge(ie_df, on="feature", how="outer")
        
        if importance_df is None:
            importance_df = df
        else:
            importance_df = importance_df.merge(df, on="feature", how="outer")

    return all_results, metrics_df, importance_df



In [ ]:
#Hyper parameter tuning params
# Note: improve tree functino (takes random search with arguements)
xgb_params = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [2, 3, 4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "subsample": [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 10],
    "gamma": [0, 0.1, 0.3, 0.5],
    #"reg_alpha": [0, 0.01, 0.1, 1],
    #"reg_lambda": [0.1, 1, 5, 10],
}
alphas = np.logspace(-10, 5, 300)


In [ ]:
n_iterations = 10
n_samples = 500
data_type = "poly"
op_col_name = "OP_AA_nmol_min_m3"
path_to_csv = os.path.join(".data")

for i in range(n_iterations):
    print("Iteration", i)
    df, (str_cols_names, conc_cols_names, op_cols_names, decoy_cols_names) = make_data(500, n_decoys=3, rng_seed=i)
    df, poly_cols_names = poly_expand_add_data(df, conc_cols_names + decoy_cols_names, op_cols_names)
    all_data = df, (str_cols_names, conc_cols_names, op_cols_names, decoy_cols_names, poly_cols_names)
    split_data = random_split_helper(all_data, data_type, op_col_name, random_seed=i)

    lin_models = {
    "linear": LinearRegression(n_jobs=-1),
    "lasso": LassoCV(alphas=alphas, max_iter=10000, tol=1e-6, n_jobs=-1, random_state=i),
    "elastic": ElasticNetCV(alphas=alphas, l1_ratio=np.linspace(0, 1, 50), max_iter=10000, n_jobs=-1, random_state=i)
    }

    tree_models = {
        "XGboost": (RandomizedSearchCV(XGBRegressor(),param_distributions=xgb_params, refit=True, n_jobs=-1, random_state=i))
        }

    all_results, metrics, coefs = benchmark(split_data, lin_models, tree_models)

    metrics.to_csv(os.path.join(path_to_csv, f"metrics_{op_col_name}_{data_type}_{i}.csv"))
    coefs.to_csv(os.path.join(path_to_csv, f"coefs_{op_col_name}_{data_type}_{i}.csv"))
    all_results["XGboost"]["int_shap_mean_abs_pairwise"].to_csv(os.path.join(path_to_csv, f"intshap_{op_col_name}_{data_type}_{i}.csv"))

